## Setup

In [1]:
from src.py_src.models import SolarFlarePredictor
from src.py_src import util
from src.py_src.output import evaluation as ev
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
predictor = SolarFlarePredictor(windows=['24h'])

slided_df_path = os.path.join(os.getenv("SLIDED_DFS_CSV_PATH"), "data_slided_V4.parquet")
target_class = 'target_class_in_24h'
target_flux = 'target_flux_in_24h'
target_columns = [target_class, target_flux]

df_model_input = util.create_df_model_input_opt(slided_df_path, target_columns, "xl_")

train_pct = 0.7
val_pct = (1-train_pct)/2

data = util.prepare_data(
    df_model_input=df_model_input,
    target_class_col=target_class,
    train_pct=train_pct,
    val_pct=val_pct,
    target_flux_col=target_flux,
    lambda_function= lambda lb: lb
)

X_test = data['x']['test']
y_test = data['y']['test']

Carregando 46 colunas do arquivo Parquet...


In [3]:
backtester = ev.SolarfallBacktester(predictor, window_name='24h')

df_results = backtester.simulate_cascade(X_test, y_test)
df_results

--- Executando Simulação da Cascata (Vetorizada) ---
Calculando predições brutas...
Consolidando Decisões Hierárquicas...


,Predicted_Class,Actual_Class,Is_Correct,Raw_GK,Raw_MX_LogFlux
ds,,,,,
2020-08-23 12:00:00,No Flare,No Flare,True,0,-4.752018
2020-08-23 12:10:00,No Flare,No Flare,True,0,-4.752018
2020-08-23 12:20:00,No Flare,No Flare,True,0,-4.752018
2020-08-23 12:30:00,No Flare,No Flare,True,0,-4.752018
2020-08-23 12:40:00,No Flare,No Flare,True,0,-4.752018
...,...,...,...,...,...
2024-12-28 23:10:00,Class M,Class X,False,1,-4.401088
2024-12-28 23:20:00,Class M,Class X,False,1,-4.446469
2024-12-28 23:30:00,Class M,Class X,False,1,-4.463571


In [4]:
df_metrics, df_transitions = backtester.get_performance_dataframes()

In [5]:
df_metrics

,Total Real,Total Predito,Acertos,Falsos Negativos (Omissão),Falsos Positivos (Alarme),Precisão (%),Recall (%)
Classe,,,,,,,
No Flare,19833.0,19662.0,13394.0,0.0,0.0,68.121249,67.533908
Class A/B,37546.0,52918.0,28300.0,5752.0,5866.0,53.478967,75.374208
Class C,92642.0,83199.0,55476.0,17899.0,3853.0,66.678686,59.882127
Class M,69773.0,65897.0,41962.0,23683.0,18645.0,63.678164,60.140742
Class X,8950.0,7068.0,2104.0,6846.0,4964.0,29.767968,23.508380
TOTAL SISTEMA,228744.0,228744.0,141236.0,54180.0,33328.0,61.744133,61.744133


In [6]:
df_transitions

Predicted_Class,No Flare,Class A/B,Class C,Class M,Class X,Total Real
Actual_Class,,,,,,
No Flare,13394,5866,436,137,0,19833
Class A/B,5752,28300,3417,77,0,37546
Class C,516,17383,55476,18431,836,92642
Class M,0,1290,22393,41962,4128,69773
Class X,0,79,1477,5290,2104,8950
